In [6]:
import os
import json
from typing import Literal

from dotenv import load_dotenv
from openai import OpenAI
from helpers.prompt_maker import get_links_system_prompt, get_links_user_prompt

In [7]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

anthropic_url = "https://api.anthropic.com/v1"
ollama_url = "http://localhost:11434/v1"

openai = OpenAI(api_key=openai_api_key)
anthropic = OpenAI(base_url=anthropic_url, api_key=anthropic_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [8]:
def define_model_and_provider(
    model_type: Literal["gpt", "claude", "llama"],
) -> tuple[str, OpenAI]:
    if model_type.lower() == "gpt":
        model = "gpt-5.6-luna"
        provider = openai
    elif model_type.lower() == "claude":
        model = "claude-sonnet-5"
        provider = anthropic
    else:
        model = "llama3.2"
        provider = ollama
    return model, provider

In [13]:
def chat(
    system_prompt: str,
    user_prompt: str,
    model_type: Literal["gpt", "claude", "llama"] = "gpt",
    json=False,
) -> str:
    model, provider = define_model_and_provider(model_type)
    response = provider.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format={"type": "json_object"} if json else {"type": "text"},
    )
    result = response.choices[0].message.content or ""
    return result

In [16]:
def select_relevant_links(url: str, model_type: Literal["gpt", "claude", "llama"]):
    system_prompt = get_links_system_prompt()
    user_prompt = get_links_user_prompt(url)
    result = chat(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        model_type=model_type,
        json=True,
    )
    links = json.loads(result)
    return links

In [17]:
print(select_relevant_links("https://www.anthropic.com/", "llama"))

{'links': [{'type': 'Company page', 'url': 'https://www.anthropic.com/'}, {'type': 'About the product Claude', 'url': 'https://claude.com/product/overview'}, {'type': "Overview of the company's mission", 'url': 'https://x.com/AnthropicAI'}, {'type': 'LinkedIn Company Page', 'url': 'https://www.linkedin.com/company/anthropicresearch'}, {'type': 'YouTube Channel', 'url': 'https://www.youtube.com/@anthropic-ai'}, {'type': 'Blog', 'url': 'https://www.claude.com/blog'}, {'type': 'Claude AI Platform', 'url': 'https://platform.claude.com/'}]}
